# Tutorial 01 — Your first ring attractor 🪰🧭

**Everyone in the team does this notebook (Week 1).** It takes about 1–2 hours. Do it yourself, and don't just read it!

By the end you will be able to:
1. explain what the network equation means, term by term;
2. build the three connectivity profiles from the paper (local, global, winner-take-all);
3. create a *bump* of activity and measure its position (PVA) and width;
4. show that the bump **persists in darkness** and **tracks a moving visual cue** (the paper's Fig. 1G–H, in a model).

> Tip: run cells with `Shift + Enter`. If something breaks, `Kernel → Restart & Run All`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))   # so we can import our model
import numpy as np
import matplotlib.pyplot as plt
import ring_model as rm

plt.rcParams['figure.figsize'] = (8, 3.5)

## 1. The equation

We have $N$ neurons on a ring. Neuron $i$ prefers heading $\theta_i = 2\pi i / N$. Its firing rate $u_i$ follows

$$\tau \frac{du_i}{dt} = -u_i + \Big[\, \sum_j W_{ij}\, u_j \; + \; b \; + \; I_i(t) \Big]_+$$

| term | meaning | biology |
|---|---|---|
| $-u_i$ | leak: activity decays without drive | membrane / calcium decay |
| $\sum_j W_{ij} u_j$ | recurrent input from other ring neurons | E-PG ↔ P-EN / ring neuron circuits |
| $b = 1$ | constant background drive | tonic input |
| $I_i(t)$ | external input | visual landmark, or the optogenetic laser |
| $[x]_+ = \max(x,0)$ | firing rates can't be negative | threshold |

We measure time in units of $\tau$, so $\tau = 1$.

**✏️ Question for the team:** which term would you change to model "the fly is in darkness"?

## 2. Connectivity: who talks to whom?

$W_{ij}$ only depends on the angular *distance* between neurons $i$ and $j$. So we can plot it as one curve: "weight vs distance".

In [ ]:
N = 64
th = rm.ring_angles(N)
dist = rm.wrap_angle(th)               # distance from neuron 0 to every other neuron
order = np.argsort(dist)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, model in zip(axes, ['local', 'global', 'wta']):
    W = rm.make_connectivity(model, N=N)
    ax.plot(np.degrees(dist[order]), W[0, order], '.-')
    ax.axhline(0, color='k', lw=0.5)
    ax.set_title(f'{model} model')
    ax.set_xlabel('distance between neurons (deg)')
axes[0].set_ylabel('weight $W_{0j}$')
plt.tight_layout()

**What to notice**
- **Local**: a big spike at distance 0 (self + nearest neighbours) and a *flat negative* level everywhere else → *local excitation, global inhibition*.
- **Global**: a smooth cosine. Close neurons excite each other, far ones inhibit.
- **WTA**: self-excitation, everyone else inhibited equally. There are no "neighbours" at all.

**✏️ Exercise 2.1:** In `make_connectivity('local', N=N, alpha=..., beta=..., D=...)` change `beta` (inhibition) to 10 and 40. How does the curve change? (Come back to this after section 3 and check how the bump changes.)

## 3. Make a bump

We put a tiny seed of activity at angle 0 and let the network do the rest.

In [ ]:
W = rm.make_connectivity('local', N=N)
u0 = rm.initial_bump(N, center=0.0)
t, U = rm.simulate(W, u0, inputs=np.zeros(N), T=20, dt=0.005, record_every=20)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].imshow(np.roll(U, N//2, axis=1).T, aspect='auto', origin='lower',
               extent=[t[0], t[-1], -180, 180], cmap='Blues')
axes[0].set_xlabel(r'time ($\tau$)'); axes[0].set_ylabel('neuron (deg)')
axes[0].set_title('activity over time')
axes[1].plot(np.degrees(rm.wrap_angle(th)[order]), U[-1][order], '.-')
axes[1].set_xlabel('neuron preferred angle (deg)'); axes[1].set_ylabel('rate')
axes[1].set_title('final bump')
plt.tight_layout()

u_bump = U[-1]
pva_angle, pva_amp = rm.population_vector(u_bump)
print(f'number of bumps : {rm.count_bumps(u_bump)}')
print(f'bump width      : {np.degrees(rm.bump_width(u_bump)):.0f} deg   (paper: ~90 deg, Fig. 1J)')
print(f'PVA position    : {np.degrees(pva_angle):.1f} deg')

### The Population Vector Average (PVA)
The paper estimates bump position by turning every neuron into an arrow (direction = its preferred angle, length = its activity) and adding the arrows (Fig. 1F). This is `rm.population_vector`. **✏️ Exercise 3.1:** write the PVA yourself in 3 lines with `np.cos`, `np.sin` and `np.arctan2`, and check you get the same number.

In [ ]:
# Exercise 3.1 — your PVA here
# x = ...
# y = ...
# my_angle = np.arctan2(y, x)


**✏️ Exercise 3.2:** repeat the simulation with `'global'` and `'wta'`. Compare bump shape, width and peak height. Which looks most like the bump in the paper's Fig. 1E/G?

## 4. Tracking a visual cue, then darkness (model version of Fig. 1G–H)

The fly sees a bright bar. When the fly turns, the bar moves around it. We model the bar as a **weak, broad input** whose position rotates slowly. After 60 τ we switch the lights off (input = 0).

Prediction for a ring attractor: the bump follows the cue, then **stays where it was** in darkness.

In [ ]:
W = rm.make_connectivity('local', N=N)
u_bump = rm.settle_bump(W, center=0.0)
peak = u_bump.max()
speed = 0.04            # rad per tau  (how fast the "fly" turns)
t_dark = 60.0           # lights off at this time

def visual_input(t):
    if t >= t_dark:
        return np.zeros(N)                     # darkness
    cue_position = speed * t
    return 1.0 * peak * rm.von_mises(th, cue_position, 1 / 1.0**2)

t, U = rm.simulate(W, u_bump, visual_input, T=100, dt=0.005, record_every=20)
pva, amp = rm.population_vector(U)
cue = np.where(t < t_dark, rm.wrap_angle(speed * t), np.nan)

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].imshow(np.roll(U, N//2, axis=1).T, aspect='auto', origin='lower',
               extent=[t[0], t[-1], -np.pi, np.pi], cmap='Blues')
axes[0].set_ylabel('neuron (rad)')
axes[0].axvspan(t_dark, t[-1], color='k', alpha=0.08)
axes[0].text(t_dark + 1, 2.6, 'darkness')
axes[1].plot(t, pva, 'r', label='PVA estimate (bump)')
axes[1].plot(t, cue, 'b', label='visual cue')
axes[1].set_ylabel('heading (rad)'); axes[1].set_xlabel(r'time ($\tau$)')
axes[1].legend(); axes[1].axvspan(t_dark, t[-1], color='k', alpha=0.08)
plt.tight_layout()
print(f'bump position at lights-off: {pva[t <= t_dark][-1]:.2f} rad,  at the end: {pva[-1]:.2f} rad')

**What to notice:** the bump lags slightly behind the cue (it needs time to *flow*), and in darkness it does not decay or move. This is the property the paper calls **persistence**.

**✏️ Exercise 4.1:** make the cue faster (`speed = 0.2`). Can the bump keep up? What happens?

**✏️ Exercise 4.2:** in the paper the bump in darkness *slowly drifts*. Our model doesn't, because it has no noise. Add `noise_std=0.04*peak` to `rm.simulate` and look again. (This is the starting point for the Fig. 3 task.)

## 5. Check your understanding (discuss in the team meeting)
1. Why does a ring attractor need *both* excitation and inhibition?
2. What does "the bump is unique" mean, and which connectivity term enforces it?
3. In one sentence, what is the difference between a bump that **flows** and one that **jumps**? (Preview of Fig. 4. Run `python scripts/fig4_jump_vs_flow.py --quick`.)
4. Why is the PVA a good read-out, and when could it be misleading? (Hint: two bumps at once.)

✅ When you're done, tick *"Tutorial 01 done"* for your name in `PLAN.md` and commit.